# 01 — Data validation and feature engineering

This notebook prepares the historical modelling table used by the rest of the project.

It performs data validation, modern-era filtering, leakage-safe five-match form, tournament context, chronological split checks and pre-match Elo calculation. It writes reusable artefacts to disk so later notebooks do not need to repeat the historical preprocessing.

In [ ]:
from pathlib import Path
import hashlib
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

# Resolve the project root whether Jupyter starts from the repository root
# or from inside the notebooks directory.
current_dir = Path.cwd()
if (current_dir / "data" / "results.csv").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "results.csv").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate data/results.csv. Start Jupyter from the project root "
        "or the notebooks directory."
    )

results_path = project_root / "data" / "results.csv"
outputs_dir = project_root / "outputs"
outputs_dir.mkdir(exist_ok=True)

required_columns = [
    "date", "home_team", "away_team", "home_score", "away_score",
    "tournament", "city", "country", "neutral"
]

df_raw = pd.read_csv(results_path)
missing_columns = sorted(set(required_columns) - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"results.csv is missing required columns: {missing_columns}")

source_rows = len(df_raw)
df_matches = df_raw[required_columns].copy()
df_matches["date"] = pd.to_datetime(df_matches["date"], errors="coerce")
df_matches["home_score"] = pd.to_numeric(df_matches["home_score"], errors="coerce")
df_matches["away_score"] = pd.to_numeric(df_matches["away_score"], errors="coerce")

invalid_date_rows = int(df_matches["date"].isna().sum())
missing_score_rows = int(df_matches[["home_score", "away_score"]].isna().any(axis=1).sum())
exact_duplicate_rows = int(df_matches.duplicated(subset=required_columns, keep="first").sum())

# Keep only completed, valid, exact-unique matches.
df_matches = (
    df_matches
    .dropna(subset=["date", "home_score", "away_score"])
    .drop_duplicates(subset=required_columns, keep="first")
    .copy()
)

# Deterministic match identifier derived from the complete validated record.
def make_match_id(row):
    values = []
    for column in required_columns:
        value = row[column]
        if column == "date":
            value = value.strftime("%Y-%m-%d")
        elif isinstance(value, float) and value.is_integer():
            value = int(value)
        values.append(str(value).strip())
    payload = "|".join(values)
    return "match_" + hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

df_matches["match_id"] = df_matches.apply(make_match_id, axis=1)
if not df_matches["match_id"].is_unique:
    raise ValueError("Generated match_id values are not unique.")

# Report keys that are not guaranteed unique even though complete records differ.
duplicate_key_mask = df_matches.duplicated(
    subset=["date", "home_team", "away_team"], keep=False
)
duplicate_match_keys = df_matches.loc[
    duplicate_key_mask,
    ["match_id", "date", "home_team", "away_team", "home_score", "away_score", "tournament"]
].sort_values(["date", "home_team", "away_team"])
duplicate_match_keys.to_csv(outputs_dir / "duplicate_match_keys.csv", index=False)

validation_summary = pd.DataFrame([
    ("source_rows", source_rows),
    ("invalid_date_rows", invalid_date_rows),
    ("missing_score_rows_removed", missing_score_rows),
    ("exact_duplicate_rows_removed", exact_duplicate_rows),
    ("validated_completed_matches", len(df_matches)),
    ("unique_match_ids", df_matches["match_id"].nunique()),
    ("duplicate_date_home_away_rows", len(duplicate_match_keys)),
], columns=["metric", "value"])
validation_summary.to_csv(outputs_dir / "data_validation_summary.csv", index=False)

print("Project root:", project_root)
print("Dataset path:", results_path)
print("Validated completed matches:", len(df_matches))
print("Unique match IDs:", df_matches["match_id"].nunique())
display(validation_summary)
display(df_matches.head())

In [ ]:
# Filter to the modern modelling period after validation.
modern_era_cutoff = pd.Timestamp("2000-01-01")
df_clean = df_matches.loc[df_matches["date"] >= modern_era_cutoff].copy()

if not df_clean["match_id"].is_unique:
    raise ValueError("match_id must remain unique after filtering.")

print("Validated completed matches:", len(df_matches))
print("Total matches from 2000 onward:", len(df_clean))
print("Unique modern-era match IDs:", df_clean["match_id"].nunique())
print("\nTop 10 Tournament Types:")
print(df_clean["tournament"].value_counts().head(10))

In [ ]:
# Build one team-perspective row for each side of every validated match.
df_home = df_clean[[
    "match_id", "date", "home_team", "home_score", "away_score"
]].copy()
df_home["side"] = "home"
df_home = df_home.rename(columns={
    "home_team": "team",
    "home_score": "goals_for",
    "away_score": "goals_against",
})

df_away = df_clean[[
    "match_id", "date", "away_team", "away_score", "home_score"
]].copy()
df_away["side"] = "away"
df_away = df_away.rename(columns={
    "away_team": "team",
    "away_score": "goals_for",
    "home_score": "goals_against",
})

df_team_matches = (
    pd.concat([df_home, df_away], ignore_index=True)
    .sort_values(["team", "date", "match_id", "side"])
    .reset_index(drop=True)
)

expected_team_rows = 2 * len(df_clean)
if len(df_team_matches) != expected_team_rows:
    raise ValueError(
        f"Expected {expected_team_rows} team-perspective rows, found {len(df_team_matches)}."
    )

# Previous-five-match rolling form. shift(1) excludes the current result.
window_size = 5
df_team_matches["form_goals_for"] = (
    df_team_matches.groupby("team")["goals_for"]
    .transform(lambda s: s.shift(1).rolling(window_size, min_periods=1).mean())
)
df_team_matches["form_goals_against"] = (
    df_team_matches.groupby("team")["goals_against"]
    .transform(lambda s: s.shift(1).rolling(window_size, min_periods=1).mean())
)

# If one team has multiple source matches on the same date, no kickoff times
# are available. Give every same-day row the pre-date value from the first row
# so one same-day result cannot leak into another.
df_team_matches["form_goals_for"] = (
    df_team_matches.groupby(["team", "date"])["form_goals_for"].transform("first")
)
df_team_matches["form_goals_against"] = (
    df_team_matches.groupby(["team", "date"])["form_goals_against"].transform("first")
)

df_team_matches[["form_goals_for", "form_goals_against"]] = (
    df_team_matches[["form_goals_for", "form_goals_against"]].fillna(0.0)
)

same_day_rows = df_team_matches[
    df_team_matches.duplicated(["team", "date"], keep=False)
]
same_day_form_counts = same_day_rows.groupby(["team", "date"])[
    ["form_goals_for", "form_goals_against"]
].nunique()
if not same_day_form_counts.empty and (same_day_form_counts > 1).any().any():
    raise ValueError("Same-day matches received different pre-date form values.")

print("Team-perspective rows:", len(df_team_matches))
print("Expected team-perspective rows:", expected_team_rows)
print("Same-day team/date groups handled:", len(same_day_form_counts))
print("\nArgentina's leakage-safe form check:")
display(df_team_matches.loc[df_team_matches["team"].eq("Argentina")].tail(10))

In [ ]:
# Create one unambiguous form lookup for each side of every match.
home_form = (
    df_team_matches.loc[
        df_team_matches["side"].eq("home"),
        ["match_id", "form_goals_for", "form_goals_against"]
    ]
    .rename(columns={
        "form_goals_for": "home_form_goals_for",
        "form_goals_against": "home_form_goals_against",
    })
)

away_form = (
    df_team_matches.loc[
        df_team_matches["side"].eq("away"),
        ["match_id", "form_goals_for", "form_goals_against"]
    ]
    .rename(columns={
        "form_goals_for": "away_form_goals_for",
        "form_goals_against": "away_form_goals_against",
    })
)

if not home_form["match_id"].is_unique or not away_form["match_id"].is_unique:
    raise ValueError("Rolling-form lookups must contain one row per match_id.")

df_model = (
    df_clean
    .merge(home_form, on="match_id", how="left", validate="one_to_one")
    .merge(away_form, on="match_id", how="left", validate="one_to_one")
)

form_columns = [
    "home_form_goals_for", "home_form_goals_against",
    "away_form_goals_for", "away_form_goals_against",
]
if df_model[form_columns].isna().any().any():
    raise ValueError("Missing rolling-form values after match_id merge.")
if len(df_model) != len(df_clean) or not df_model["match_id"].is_unique:
    raise ValueError("Rolling-form merge changed the one-row-per-match structure.")

conditions = [
    df_model["home_score"] > df_model["away_score"],
    df_model["home_score"] == df_model["away_score"],
    df_model["home_score"] < df_model["away_score"],
]
df_model["target"] = np.select(conditions, [2, 1, 0], default=np.nan)
if df_model["target"].isna().any():
    raise ValueError("Target encoding produced missing values.")

rolling_form_summary = pd.DataFrame([
    ("modern_matches", len(df_clean)),
    ("team_perspective_rows", len(df_team_matches)),
    ("expected_team_perspective_rows", 2 * len(df_clean)),
    ("same_day_team_date_groups", len(same_day_form_counts)),
    ("home_form_lookup_rows", len(home_form)),
    ("away_form_lookup_rows", len(away_form)),
    ("final_model_rows", len(df_model)),
    ("unique_model_match_ids", df_model["match_id"].nunique()),
    ("duplicate_model_match_ids", int(df_model["match_id"].duplicated().sum())),
    ("rows_added_or_lost", len(df_model) - len(df_clean)),
    ("missing_form_values", int(df_model[form_columns].isna().sum().sum())),
], columns=["metric", "value"])
rolling_form_summary.to_csv(outputs_dir / "rolling_form_validation.csv", index=False)

print("Final feature matrix rows:", len(df_model))
print("Unique match IDs:", df_model["match_id"].nunique())
print("Rows added or lost:", len(df_model) - len(df_clean))
display(rolling_form_summary)

In [ ]:
# Correct tournament importance and match-context features.
def normalize_tournament_name(tournament_name):
    # casefold handles case consistently; NFKD normalization removes accents,
    # so "Copa América" and "Copa America" are treated identically.
    text = str(tournament_name).casefold()
    return "".join(
        character for character in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(character)
    )


def get_tournament_weight(tournament_name):
    name = normalize_tournament_name(tournament_name)

    if "fifa world cup" in name and "qualification" not in name:
        return 1.0

    # Preserve the original prototype's intended high-weight competitions,
    # but match them explicitly so unrelated strings containing "Euro" are
    # not accidentally treated as UEFA European Championship finals.
    major_finals = (
        "confederations cup",
        "copa america",
        "uefa euro",
    )
    if any(term in name for term in major_finals) and "qualification" not in name:
        return 0.8

    if "qualification" in name:
        return 0.6
    if "nations league" in name:
        return 0.5
    if "friendly" in name:
        return 0.25
    return 0.4


def get_original_tournament_weight(tournament_name):
    # Reproduce the pre-Commit-5 rules for validation only.
    if "FIFA World Cup" in tournament_name and "qualification" not in tournament_name:
        return 1.0
    elif (
        "Confederations Cup" in tournament_name
        or "Copa America" in tournament_name
        or "Euro" in tournament_name and "qualification" not in tournament_name
    ):
        return 0.8
    elif "qualification" in tournament_name:
        return 0.6
    elif "Nations League" in tournament_name:
        return 0.5
    elif "Friendly" in tournament_name:
        return 0.25
    return 0.4


# Add context directly to the one-row-per-match modelling table. Do not rebuild
# the rolling-form merge.
df_model["match_weight"] = df_model["tournament"].apply(get_tournament_weight)
df_model["is_neutral"] = df_model["neutral"].astype(int)
df_clean["match_weight"] = df_clean["tournament"].apply(get_tournament_weight)
df_clean["is_neutral"] = df_clean["neutral"].astype(int)

if len(df_model) != len(df_clean) or not df_model["match_id"].is_unique:
    raise ValueError("Match-context features changed the one-row-per-match structure.")
if not set(df_model["is_neutral"].unique()).issubset({0, 1}):
    raise ValueError("Neutral-venue feature must contain only 0 and 1.")

# Validate the bug fix against the actual dataset.
weight_audit = df_clean[["tournament"]].copy()
weight_audit["original_weight"] = weight_audit["tournament"].apply(get_original_tournament_weight)
weight_audit["corrected_weight"] = weight_audit["tournament"].apply(get_tournament_weight)

weight_validation = (
    weight_audit.groupby(["tournament", "original_weight", "corrected_weight"])
    .size()
    .reset_index(name="modern_match_count")
    .sort_values(["corrected_weight", "modern_match_count", "tournament"], ascending=[False, False, True])
)
weight_validation["weight_changed"] = (
    weight_validation["original_weight"] != weight_validation["corrected_weight"]
)
weight_validation.to_csv(outputs_dir / "tournament_weight_validation.csv", index=False)

changed_weight_rows = int((weight_audit["original_weight"] != weight_audit["corrected_weight"]).sum())
copa_america_rows = int((
    df_clean["tournament"].map(normalize_tournament_name) == "copa america"
).sum())
copa_america_correct = int((
    (df_clean["tournament"].map(normalize_tournament_name) == "copa america")
    & (df_clean["match_weight"] == 0.8)
).sum())

context_summary = pd.DataFrame([
    ("modern_matches", len(df_clean)),
    ("rows_changed_vs_original_weight_rules", changed_weight_rows),
    ("copa_america_matches", copa_america_rows),
    ("copa_america_matches_weighted_0_8", copa_america_correct),
    ("world_cup_final_weight_1_0", int((df_clean["match_weight"] == 1.0).sum())),
    ("major_final_weight_0_8", int((df_clean["match_weight"] == 0.8).sum())),
    ("qualification_weight_0_6", int((df_clean["match_weight"] == 0.6).sum())),
    ("nations_league_weight_0_5", int((df_clean["match_weight"] == 0.5).sum())),
    ("other_weight_0_4", int((df_clean["match_weight"] == 0.4).sum())),
    ("friendly_weight_0_25", int((df_clean["match_weight"] == 0.25).sum())),
    ("neutral_matches", int(df_clean["is_neutral"].sum())),
], columns=["metric", "value"])
context_summary.to_csv(outputs_dir / "match_context_validation.csv", index=False)

if copa_america_rows != copa_america_correct:
    raise ValueError("Not all Copa América finals received weight 0.8.")

features = [
    "home_form_goals_for", "home_form_goals_against",
    "away_form_goals_for", "away_form_goals_against",
    "match_weight", "is_neutral",
]
X = df_model[features]
y = df_model["target"]

print("Corrected tournament weights added without rebuilding the feature merge.")
print("Copa América matches correctly weighted 0.8:", copa_america_correct)
print("Rows whose weight changed vs original rules:", changed_weight_rows)
print("Feature matrix shape:", X.shape)
display(context_summary)
display(weight_validation.loc[weight_validation["weight_changed"]])

In [ ]:
# Chronological forecast evaluation.
evaluation_cutoff = pd.Timestamp('2023-01-01')

train_mask = df_model['date'] < evaluation_cutoff
test_mask = df_model['date'] >= evaluation_cutoff

y_train = df_model.loc[train_mask, 'target'].copy()
y_test = df_model.loc[test_mask, 'target'].copy()

training_rows = int(train_mask.sum())
test_rows = int(test_mask.sum())

if training_rows == 0 or test_rows == 0:
    raise ValueError('Chronological split produced an empty training or test set.')
if training_rows + test_rows != len(df_model):
    raise ValueError('Chronological split does not cover every modelling row exactly once.')

train_date_min = df_model.loc[train_mask, 'date'].min()
train_date_max = df_model.loc[train_mask, 'date'].max()
test_date_min = df_model.loc[test_mask, 'date'].min()
test_date_max = df_model.loc[test_mask, 'date'].max()

if train_date_max >= test_date_min:
    raise ValueError('Temporal leakage detected: training dates overlap the test period.')

split_summary = pd.DataFrame([
    ('evaluation_cutoff', evaluation_cutoff.strftime('%Y-%m-%d')),
    ('total_model_rows', len(df_model)),
    ('training_rows', training_rows),
    ('test_rows', test_rows),
    ('training_start', train_date_min.strftime('%Y-%m-%d')),
    ('training_end', train_date_max.strftime('%Y-%m-%d')),
    ('test_start', test_date_min.strftime('%Y-%m-%d')),
    ('test_end', test_date_max.strftime('%Y-%m-%d')),
    ('date_overlap_detected', int(train_date_max >= test_date_min)),
], columns=['metric', 'value'])
split_summary.to_csv(outputs_dir / 'chronological_split_validation.csv', index=False)

class_distribution = pd.DataFrame({
    'training_count': y_train.value_counts().reindex([0, 1, 2], fill_value=0),
    'test_count': y_test.value_counts().reindex([0, 1, 2], fill_value=0),
}).astype(int)
class_distribution.index.name = 'target'
class_distribution.to_csv(outputs_dir / 'chronological_split_class_distribution.csv')

print('Chronological evaluation split')
print('Training:', train_date_min.date(), 'to', train_date_max.date(), '-', training_rows, 'matches')
print('Testing: ', test_date_min.date(), 'to', test_date_max.date(), '-', test_rows, 'matches')
display(split_summary)
display(class_distribution)

In [ ]:
# Deterministic pre-match Elo features keyed by match_id.
def get_expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def elo_rating_change(rating, expected, actual, k):
    return k * (actual - expected)


# Baseline Elo for a team at its first modern-era appearance.
elo_baseline = 1500.0
current_elo = {}
elo_feature_rows = []

# Process one calendar date at a time. If a team appears more than once on
# the same date, every match on that date must see the same pre-date Elo.
# Daily rating changes are therefore accumulated and applied only after all
# matches on that date have been evaluated.
elo_source = df_clean.sort_values(["date", "match_id"]).copy()

print("Calculating leakage-safe historical Elo ratings...")
for match_date, day_matches in elo_source.groupby("date", sort=True):
    pre_date_ratings = dict(current_elo)
    daily_changes = {}

    for row in day_matches.itertuples(index=False):
        home_rating = float(pre_date_ratings.get(row.home_team, elo_baseline))
        away_rating = float(pre_date_ratings.get(row.away_team, elo_baseline))

        elo_feature_rows.append({
            "match_id": row.match_id,
            "home_elo": home_rating,
            "away_elo": away_rating,
        })

        if row.home_score > row.away_score:
            actual_home, actual_away = 1.0, 0.0
        elif row.home_score < row.away_score:
            actual_home, actual_away = 0.0, 1.0
        else:
            actual_home, actual_away = 0.5, 0.5

        expected_home = get_expected_score(home_rating, away_rating)
        expected_away = get_expected_score(away_rating, home_rating)
        k_adjusted = 30.0 * float(row.match_weight)

        home_change = elo_rating_change(
            home_rating, expected_home, actual_home, k_adjusted
        )
        away_change = elo_rating_change(
            away_rating, expected_away, actual_away, k_adjusted
        )

        daily_changes[row.home_team] = daily_changes.get(row.home_team, 0.0) + home_change
        daily_changes[row.away_team] = daily_changes.get(row.away_team, 0.0) + away_change

    # Apply the whole date batch after every match has received pre-date Elo.
    teams_today = set(day_matches["home_team"]) | set(day_matches["away_team"])
    for team in teams_today:
        starting_rating = float(pre_date_ratings.get(team, elo_baseline))
        current_elo[team] = starting_rating + daily_changes.get(team, 0.0)

elo_features = pd.DataFrame(elo_feature_rows)

if len(elo_features) != len(df_clean):
    raise ValueError(
        f"Expected {len(df_clean)} Elo feature rows, found {len(elo_features)}."
    )
if not elo_features["match_id"].is_unique:
    raise ValueError("Elo feature table must contain one row per match_id.")
if set(elo_features["match_id"]) != set(df_clean["match_id"]):
    raise ValueError("Elo feature match_id values do not match the validated data.")

# Verify repeated team/date appearances received identical pre-date ratings.
elo_audit = df_clean[["match_id", "date", "home_team", "away_team"]].merge(
    elo_features, on="match_id", how="left", validate="one_to_one"
)
home_elo_audit = elo_audit[["date", "home_team", "home_elo"]].rename(
    columns={"home_team": "team", "home_elo": "elo"}
)
away_elo_audit = elo_audit[["date", "away_team", "away_elo"]].rename(
    columns={"away_team": "team", "away_elo": "elo"}
)
team_date_elo_audit = pd.concat([home_elo_audit, away_elo_audit], ignore_index=True)
repeated_team_dates = team_date_elo_audit[
    team_date_elo_audit.duplicated(["team", "date"], keep=False)
]
repeated_rating_counts = repeated_team_dates.groupby(["team", "date"])["elo"].nunique()
if not repeated_rating_counts.empty and (repeated_rating_counts > 1).any():
    raise ValueError("Same-day matches received different pre-date Elo ratings.")

final_elo_ratings = (
    pd.DataFrame(current_elo.items(), columns=["team", "final_elo"])
    .sort_values("final_elo", ascending=False)
    .reset_index(drop=True)
)
final_elo_ratings.to_csv(outputs_dir / "final_elo_ratings.csv", index=False)

print("Elo feature rows:", len(elo_features))
print("Unique Elo match IDs:", elo_features["match_id"].nunique())
print("Repeated team/date groups handled:", len(repeated_rating_counts))
print("\nTop 5 Teams by Final Elo Rating:")
display(final_elo_ratings.head(5))

In [ ]:
# Join pre-match Elo features to the modelling table.
model_rows_before_elo = len(df_model)
df_model = df_model.merge(
    elo_features,
    on="match_id",
    how="left",
    validate="one_to_one",
)

elo_columns = ["home_elo", "away_elo"]
missing_elo_values = int(
    df_model[elo_columns].isna().sum().sum()
)

if missing_elo_values:
    raise ValueError(
        f"Elo merge produced {missing_elo_values} missing values."
    )
if len(df_model) != model_rows_before_elo:
    raise ValueError(
        "Elo merge changed the number of model rows."
    )
if not df_model["match_id"].is_unique:
    raise ValueError(
        "df_model contains duplicate match_id values after Elo merge."
    )

elo_validation = pd.DataFrame([
    ("modern_matches", len(df_clean)),
    ("elo_feature_rows", len(elo_features)),
    ("unique_elo_match_ids", elo_features["match_id"].nunique()),
    ("model_rows_before_elo_merge", model_rows_before_elo),
    ("model_rows_after_elo_merge", len(df_model)),
    ("duplicate_model_match_ids", int(df_model["match_id"].duplicated().sum())),
    ("missing_elo_values", missing_elo_values),
    ("repeated_team_date_groups", len(repeated_rating_counts)),
    ("teams_with_final_elo", len(final_elo_ratings)),
], columns=["metric", "value"])
elo_validation.to_csv(
    outputs_dir / "elo_validation.csv",
    index=False,
)

feature_columns = [
    "home_elo",
    "away_elo",
    "home_form_goals_for",
    "home_form_goals_against",
    "away_form_goals_for",
    "away_form_goals_against",
    "match_weight",
    "is_neutral",
]

# Current team state used by future/hypothetical fixtures.
elo_state = pd.DataFrame(
    current_elo.items(),
    columns=["team", "current_elo"],
)

recent_completed_matches = (
    df_team_matches
    .sort_values(["team", "date", "match_id", "side"])
    .groupby("team", group_keys=False)
    .tail(window_size)
)

current_form = (
    recent_completed_matches
    .groupby("team", as_index=False)
    .agg(
        current_form_goals_for=("goals_for", "mean"),
        current_form_goals_against=("goals_against", "mean"),
        form_matches_used=("match_id", "count"),
        latest_match_date=("date", "max"),
    )
)

team_state_snapshot = (
    elo_state
    .merge(
        current_form,
        on="team",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "current_elo",
        ascending=False,
    )
    .reset_index(drop=True)
)

if team_state_snapshot["team"].duplicated().any():
    raise ValueError(
        "Team-state snapshot contains duplicate team names."
    )
if team_state_snapshot[[
    "current_elo",
    "current_form_goals_for",
    "current_form_goals_against",
]].isna().any().any():
    raise ValueError(
        "Team-state snapshot contains missing model inputs."
    )

team_state_snapshot.to_csv(
    outputs_dir / "team_state_snapshot.csv",
    index=False,
)

intermediate_dir = outputs_dir / "intermediate"
intermediate_dir.mkdir(exist_ok=True)

df_model.to_csv(
    intermediate_dir / "model_features.csv",
    index=False,
)

import json

data_pipeline_metadata = {
    "feature_columns": feature_columns,
    "evaluation_cutoff": evaluation_cutoff.strftime("%Y-%m-%d"),
    "training_rows": training_rows,
    "test_rows": test_rows,
    "training_start": train_date_min.strftime("%Y-%m-%d"),
    "training_end": train_date_max.strftime("%Y-%m-%d"),
    "holdout_start": test_date_min.strftime("%Y-%m-%d"),
    "holdout_end": test_date_max.strftime("%Y-%m-%d"),
    "state_data_end": df_clean["date"].max().strftime("%Y-%m-%d"),
    "modern_model_rows": len(df_model),
    "team_state_count": len(team_state_snapshot),
}

(intermediate_dir / "data_pipeline_metadata.json").write_text(
    json.dumps(
        data_pipeline_metadata,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Prepared modelling artefact:",
    intermediate_dir / "model_features.csv",
)
print(
    "Frozen team-state rows:",
    len(team_state_snapshot),
)